In [ ]:
import pandas as pd
from openai import OpenAI
import json

# --- User Configuration ---
csv_file_path = '/content/drive/MyDrive/Preperation Phase/Code/scopus_export_Mar 17-2026_1443c7fc-4736-4a87-991d-3e0efdd9aba9.csv' # <<<--- UPDATE THIS with the path to your CSV file
abstract_column_name = 'abstract' # <<<--- UPDATE THIS with the name of the column containing abstracts

# Assuming openai_api_key is defined in a previous cell
openai_api_key="xxxx"

client = OpenAI(api_key=openai_api_key)

def evaluate_paper(title,abstract):
    if not abstract or pd.isna(abstract):
        return {'included': False, 'reasoning': 'Abstract is empty or missing.'}

    # Define the criteria for the LLM
    criteria_prompt = f"""You are an expert in software engineering research paper classification. Your task is to evaluate a research paper's abstract against specific inclusion and exclusion criteria related to  PTMs/LLMs, and foundation models categorization for software engineering tasks.
Here are the criteria:

**Inclusion Criteria:**
*   **IC1:** Discusses PTMs, LLMs, or foundation models applied to software engineering (prerequisite)
*   **IC2:** Categorizes SE tasks, or explicitly maps PTMs/LLMs to SE tasks
*   **IC3:** Discusses model suitability, capability, or selection rationale for specific SE tasks

**Exclusion Criteria:**
*   **EC1:** Focus exclusively on non-SE domains (e.g., general NLP, computer vision, biomedical)
*   **EC2:** Are benchmark or dataset papers, or performance-only studies, with no task categorization or model–task relationship discussion
*   **EC3:** Focus on SE tasks in learning and education contexts (e.g., programming education, teaching tools)
*   **EC4:** Focus on LLM-based agents in SE

**Decision Rule:**
Include papers that satisfy IC1 AND (at least one of IC2 or IC3).
Exclude papers that meet any of EC1, EC2, EC3, or EC4, even if they satisfy inclusion criteria.

**Your response must be a JSON object with two keys:**
*   `included`: a boolean (true if the paper should be included based on the decision rule, false otherwise).
*   `reasoning`: a string explaining *why* the paper was included or excluded, referencing the specific criteria.

Here is the abstract to evaluate:
---
{abstract}
---
"""

    try:
        completion = client.chat.completions.create(
            model="gpt-4o", # You can choose another suitable model like gpt-3.5-turbo if preferred
            response_format={ "type": "json_object" },
            messages=[
                {"role": "system", "content": "You are an expert in classifying research papers based on provided criteria. Respond only with a JSON object as specified."},
                {"role": "user", "content": criteria_prompt}
            ]
        )
        response_content = completion.choices[0].message.content
        result = json.loads(response_content)
        print(result)
        return result
    except Exception as e:
        print(f"Error processing abstract: {e}")
        return {'included': False, 'reasoning': f'API call failed: {e}'}

# Load the CSV file
try:
    df = pd.read_csv(csv_file_path)
    print(f"Successfully loaded {len(df)} papers from {csv_file_path}.")
except FileNotFoundError:
    print(f"Error: The file '{csv_file_path}' was not found. Please check the path.")
    df = pd.DataFrame() # Create an empty DataFrame to prevent further errors

if not df.empty and abstract_column_name in df.columns:
    # Apply the evaluation function to each abstract
    # This might take a while depending on the number of papers and API latency
    print("Starting paper evaluation using OpenAI API...")
    df['evaluation_result'] = df[abstract_column_name].apply(evaluate_paper)

    # Extract 'included' status and 'reasoning'
    df['should_include'] = df['evaluation_result'].apply(lambda x: x['included'])
    df['evaluation_reasoning'] = df['evaluation_result'].apply(lambda x: x['reasoning'])

    # Filter papers that should be included
    filtered_papers = df[df['should_include'] == True]

    print(f"\nEvaluation complete. Found {len(filtered_papers)} papers matching the criteria.")
    print("\nFirst 5 filtered papers (with reasoning):")
    display(filtered_papers[['title', abstract_column_name, 'evaluation_reasoning']].head())
else:
    if not df.empty:
        print(f"Error: Column '{abstract_column_name}' not found in the CSV file. Available columns: {df.columns.tolist()}")
    print("No papers to process or DataFrame is empty.")


Successfully loaded 549 papers from /content/drive/MyDrive/Preperation Phase/Code/scopus_export_Mar 17-2026_1443c7fc-4736-4a87-991d-3e0efdd9aba9.csv.
Starting paper evaluation using OpenAI API...
{'included': False, 'reasoning': "The abstract lists several papers, but none clearly discuss PTMs, LLMs, or foundation models specifically applied to software engineering tasks in a manner that satisfies IC1. One paper involves large language models in context of secure smart contract code generation, but there's no explicit mention of categorizing SE tasks or discussing suitability rationale (IC2 or IC3). Additionally, the context appears to align with EC1 or EC2 as there is a focus on specific implementations or performance, with no clear model–task relationship discussion."}
{'included': False, 'reasoning': 'The paper should be excluded because it fails to meet the prerequisite inclusion criterion IC1; it does not discuss PTMs, LLMs, or foundation models applied to software engineering tas

KeyError: 'included'

In [ ]:
{'included': False, 'reasoning': "The abstract lists several papers, but none clearly discuss PTMs, LLMs, or foundation models specifically applied to software engineering tasks in a manner that satisfies IC1. One paper involves large language models in context of secure smart contract code generation, but there's no explicit mention of categorizing SE tasks or discussing suitability rationale (IC2 or IC3). Additionally, the context appears to align with EC1 or EC2 as there is a focus on specific implementations or performance, with no clear model–task relationship discussion."}
{'included': False, 'reasoning': 'The paper should be excluded because it fails to meet the prerequisite inclusion criterion IC1; it does not discuss PTMs, LLMs, or foundation models applied to software engineering tasks. It focuses on symbolic AI and model-driven architecture for expert systems, without reference to PTMs/LLMs. Additionally, the abstract falls under EC4 as it discusses LLM-based agents, albeit tangentially, in non-specific contexts, without focus on software engineering tasks.'}
{'included': False, 'reasoning': 'The paper should be excluded based on EC4, as it focuses on the challenges faced by LLM developers, which implies a focus on LLM-based agents in software engineering. Although it discusses LLMs and their impact on software engineering (IC1), it does not categorize SE tasks or discuss model suitability or capability for specific SE tasks, hence not meeting IC2 or IC3.'}
{'included': True, 'reasoning': 'The paper satisfies IC1 as it discusses the use of LLMs, specifically GPT 3.5 Turbo and GPT 4o, applied to the software engineering task of evaluating code quality. It also fulfills IC2 by assessing and categorizing the SE task of code quality evaluation and comparing it to traditional tools like SonarQube. Additionally, it partially touches on IC3 by discussing aspects of model suitability through performance comparison, though the emphasis is more on results than on selection rationale. No exclusion criteria were met, as the focus is not on non-SE or educational contexts and it does not discuss LLM-based agents.'}
{'included': False, 'reasoning': 'The paper does not satisfy the inclusion criteria IC2 or IC3 as it does not categorize SE tasks or discuss model suitability, capability, or selection rationale for specific SE tasks. Furthermore, it meets the exclusion criteria EC2 as it appears to be a performance-only study, focusing on benchmarks and comparative analyses without discussing task categorization or the model–task relationship.'}
{'included': False, 'reasoning': 'The paper is excluded because it focuses on LLM-based agents in software engineering (EC4), which is not aligned with the inclusion criteria. While the paper discusses LLMs in relation to code readability, it does not explicitly categorize SE tasks or discuss model suitability for specific tasks (IC2 or IC3), focusing instead on code readability and perspective surveys.'}
{'included': False, 'reasoning': 'The abstract does not satisfy IC1 as it does not discuss PTMs, LLMs, or foundation models applied to software engineering. It also violates EC1, as it focuses primarily on non-SE domains such as sustainable development, IoT, and general computing technologies. Additionally, EC3 applies as one of the topics includes an educational context in software engineering. Therefore, the paper is excluded.'}
{'included': False, 'reasoning': 'The paper is excluded based on EC3 and EC4. It focuses on SE tasks in education contexts, particularly software engineering education for detecting cheating and plagiarism (EC3). Despite discussing a model built on CodeBERT, it aligns more with educational and oversight purposes rather than mapping PTMs/LLMs to a broad range of SE tasks. Additionally, it discusses the use of LLM-based tools in SE (EC4). It does not satisfy IC2 or IC3 as it lacks discussion on model capability or selection rationale for specific SE tasks beyond education-specific applications.'}
{'included': False, 'reasoning': 'The abstract mentions multiple studies and topics, but nothing specifically meeting IC1 (application of PTMs or LLMs to SE). There are no indications of task categorization (IC2), model suitability, or rationale (IC3) for SE tasks. Several topics, such as chatbots and documentations, do not focus on LLMs/PTMs for SE tasks. Additionally, topics like energy efficiency and browser automation are more aligned with EC1, which excludes studies not specific to SE applications of PTMs/LLMs.'}
{'included': False, 'reasoning': 'The paper satisfies IC1 as it discusses LLMs in the context of code development, a software engineering task. However, it does not satisfy IC2 or IC3 as it lacks explicit categorization of SE tasks or discussion on model suitability, capability, or selection rationale for specific SE tasks. Furthermore, it likely meets EC2 as it resembles a survey or tutorial focused on techniques and components without task categorization or model-task relationship discussion.'}
{'included': False, 'reasoning': 'The paper satisfies IC1 as it discusses LLMs applied to software engineering tasks, specifically comment classification. However, it does not satisfy IC2 as it does not categorize SE tasks or map PTMs/LLMs to SE tasks beyond comment classification. It also fails to satisfy IC3 as it does not discuss model suitability, capability, or selection rationale for specific SE tasks beyond the comment analysis context. Moreover, it is excluded by EC2 as it primarily focuses on performance and includes benchmark dataset contributions without extensive task categorization or model-task relationship discussions. Lastly, it could potentially be excluded under EC4 if the classified comments are considered part of LLM-based agents in SE, although that is a nuanced interpretation here.'}
{'included': False, 'reasoning': 'The abstract primarily discusses applications and challenges of AI across various fields, including healthcare, cultural heritage, and democracy, rather than focusing on software engineering. It does not satisfy IC1, as it does not discuss PTMs, LLMs, or foundation models applied specifically to software engineering tasks. Additionally, it meets EC1 for focusing on non-SE domains, such as healthcare and computational propaganda. There is no mention of categorizing SE tasks or discussing model suitability for SE tasks, thus failing IC2 and IC3 as well.'}
{'included': False, 'reasoning': 'The abstract does not explicitly discuss PTMs, LLMs, or foundation models applied to software engineering tasks, thus failing IC1. Additionally, it does not provide any explicit categorization of SE tasks or discuss model-task relationships, failing to satisfy IC2 and IC3. None of the exclusion criteria are explicitly met, but the lack of inclusion criteria results in exclusion.'}
{'included': False, 'reasoning': 'The paper meets IC1 since it discusses an LLM applied in the context of software engineering for enhancing vulnerability reports. However, it does not satisfy IC2 or IC3 as it does not categorize SE tasks or discuss model suitability, capability, or selection rationale for specific SE tasks. Additionally, it meets EC2 since the focus is on demonstrating performance (enhancing reports) without task categorization or a detailed model-task relationship discussion.'}
{'included': False, 'reasoning': "The paper mentions LLMs in relation to software-related tasks, satisfying IC1. However, it focuses on applications in educational contexts and intelligent assistance systems in resource-constrained environments, triggering EC3. Additionally, it doesn't explicitly map LLMs to SE tasks or discuss model suitability for SE tasks, lacking evidence for IC2 or IC3. Thus, it is excluded due to EC3 and partial non-satisfaction of inclusion criteria."}
{'included': False, 'reasoning': 'The paper is excluded because it focuses on LLM-based agents in SE tasks (EC4), with a specific emphasis on using ChatGPT and detecting its incorrectness in SE contexts. Although it discusses an application in software engineering, the primary focus on ChatGPT as an agent for SE tasks falls under exclusion criteria EC4, overriding the inclusion criteria.'}
{'included': False, 'reasoning': 'The paper focuses on evaluating image recognition models and uses a computer vision approach, which falls outside the scope of software engineering tasks. Therefore, it meets exclusion criterion EC1, as it is centered on a non-SE domain. None of the inclusion criteria are met, as the paper does not discuss PTMs, LLMs, or foundation models applied to software engineering tasks.'}
{'included': False, 'reasoning': 'The paper does not satisfy IC1, as it does not discuss PTMs, LLMs, or foundation models in the context of software engineering tasks. It primarily focuses on AI safety and governance in domains like power transmission and autonomous mobility, which are outside the scope of software engineering according to EC1.'}
{'included': True, 'reasoning': "The paper should be included because it satisfies the key inclusion criteria: IC1 as it explores the use of Large Language Models (LLMs) for a software engineering task (code review comment classification), IC2 by categorizing SE tasks (17 categories of code review comments), and IC3 by discussing the model's suitability and capability (LLMs outperforming state-of-the-art deep learning models). It does not meet any exclusion criteria."}
{'included': False, 'reasoning': 'The research paper does not satisfy the inclusion criteria as it does not explicitly discuss PTMs, LLMs, or foundation models applied to SE (IC1). The focus is on a machine-learning pipeline involving a Transformer-based sentence embedder and a classifier, rather than categorizing SE tasks or mapping PTMs/LLMs to SE tasks (IC2, IC3). Additionally, the paper resembles a performance study for automatic categorization without discussing model-task relationships, thus meeting EC2 for exclusion.'}
{'included': False, 'reasoning': "The abstract does not explicitly discuss PTMs, LLMs, or foundation models applied to software engineering as a primary focus (IC1 prerequisite not clearly met). While it mentions 'characterizing developers’ behaviors in LLM-supported software development,' this is one of many topics and lacks detail on whether it involves categorizing SE tasks (IC2) or discussing model suitability for specific tasks (IC3). Additionally, several topics focus on non-SE domains or educational contexts, such as programming education (EC3), indicating the paper is not focused on SE-focused applications of LLMs or PTMs."}
{'included': False, 'reasoning': 'The paper primarily describes an LLM-based framework for automating code reviews, which fulfills the prerequisite Inclusion Criterion IC1. However, it does not satisfy IC2 or IC3 as there is no explicit categorization of broader SE tasks or discussion of model suitability for various SE tasks beyond the particular focus on code review. Additionally, it fits Exclusion Criterion EC4, as it focuses on applying LLMs in a specific SE context, namely code review automation.'}
{'included': False, 'reasoning': 'The paper is excluded based on EC1 and EC2. It focuses on time-series LLMs, which, although related to LLMs, is primarily a non-software engineering domain, satisfying EC1. Additionally, the paper does not categorize software engineering tasks or discuss model-task relationships specific to software engineering, satisfying EC2. Therefore, it does not meet the inclusion criteria as it is not focused on SE tasks within the realm of PTMs/LLMs or foundation models.'}
{'included': False, 'reasoning': 'The paper mentions leveraging LLMs for software engineering tasks related to software product line (SPL) engineering, satisfying IC1. However, it does not explicitly categorize SE tasks or map PTMs/LLMs to SE tasks (IC2), nor does it discuss model suitability, capability, or selection rationale for specific SE tasks (IC3). Additionally, it fails to meet any exclusion criteria such as EC1 since it does focus on an SE domain. But, without fulfilling either IC2 or IC3, it cannot be included.'}
{'included': False, 'reasoning': 'The paper is excluded because it focuses on SE tasks in learning and education contexts (EC3) and does not map PTMs/LLMs to SE tasks outside of educational feedback, thereby violating the exclusion criterion EC3. Although it discusses LLMs, it does not satisfy the inclusion criteria for categorizing SE tasks or discussing model suitability for non-educational SE tasks.'}
{'included': False, 'reasoning': 'The paper satisfies IC1 as it discusses the use of a pre-trained model, CodeBERT, for a software engineering task (code classification). However, it does not satisfy IC2 or IC3, as there is no categorization of SE tasks or discussion on model suitability for specific SE tasks beyond performance evaluation. Additionally, it meets EC2 because it primarily focuses on performance evaluation of a model on a specific dataset, without discussing task categorization or model-task relationship beyond improving classification metrics. Thus, it is excluded based on EC2.'}
{'included': False, 'reasoning': 'The paper is excluded because it meets EC2, as it focuses mainly on datasets and model documentation rather than categorizing SE tasks or discussing model-task relationships. It doesn’t satisfy IC2 or IC3 necessary for inclusion.'}
{'included': True, 'reasoning': "The paper is included because it satisfies IC1 as it discusses LLMs applied to software engineering. It also fulfills IC2 by aiming to systematically map the applications of LLMs in SE tasks, implying a categorization effort. It doesn't meet any exclusion criteria EC1-EC4 since it's focused on SE and not on education, benchmarking, or LLM-based agents specifically."}
{'included': False, 'reasoning': 'The paper is excluded primarily based on EC2 and EC4. Although the abstract mentions the potential use of large language models (LLMs) in future directions, the study itself focuses on logging smells detection, benchmarks, and inconsistencies in detection strategies. It does not discuss PTMs/LLMs applied to current software engineering tasks, nor does it map these models to specific tasks (IC1 is not satisfied). Additionally, the paper does not discuss the model-task relationship (failing IC2 and IC3).'}
{'included': False, 'reasoning': 'The paper is excluded primarily based on EC2 and EC4. Although the abstract mentions the potential use of large language models (LLMs) in future directions, the study itself focuses on logging smells detection, benchmarks, and inconsistencies in detection strategies. It does not discuss PTMs/LLMs applied to current software engineering tasks, nor does it map these models to specific tasks (IC1 is not satisfied). Additionally, the paper does not discuss the model-task relationship (failing IC2 and IC3).'}


In [ ]:
from google.colab import drive
drive.mount('/content/drive'){'included': True, 'reasoning': 'The paper satisfies IC1 as it discusses the application of CodeBERT, a pre-trained model, to software engineering (SE) tasks. It satisfies IC2 by explicitly mapping dpCodeBERT to SE tasks such as design patterns selection and design patterns code search. While not explicitly mentioned, it implies suitability by indicating improvement in performance metrics, satisfying IC3. The paper does not meet any exclusion criteria such as EC1 through EC4.'}
{'included': True, 'reasoning': 'The paper satisfies IC1 as it discusses the application of CodeBERT, a pre-trained model, to software engineering (SE) tasks. It satisfies IC2 by explicitly mapping dpCodeBERT to SE tasks such as design patterns selection and design patterns code search. While not explicitly mentioned, it implies suitability by indicating improvement in performance metrics, satisfying IC3. The paper does not meet any exclusion criteria such as EC1 through EC4.'}


In [ ]:
import os
import json
import pandas as pd
from openai import OpenAI

# --- User Configuration ---
csv_file_path = "/content/drive/MyDrive/Preperation Phase/rr/cleaned_papers.csv"
abstract_column_name = "abstract"
title_column_name = "title"

# Better: store your API key in an environment variable instead of hardcoding it
# Example in Colab:
# import os
# os.environ["OPENAI_API_KEY"] = "your-key-here"
openai_api_key="xx"

client = OpenAI(api_key=openai_api_key)

def evaluate_paper(title, abstract):
    if pd.isna(abstract) or not str(abstract).strip():
        return {
            "included": False,
            "reasoning": "Abstract is empty or missing."
        }
def evaluate_paper(title, abstract):
    if pd.isna(abstract) or not str(abstract).strip():
        return {
            "included": False,
            "reasoning": "Abstract is empty or missing."
        }

    criteria_prompt = f"""You are an expert in software engineering research paper classification.

Your task is to evaluate a research paper's abstract based on whether it contributes to the categorization of pre-trained models (PTMs), large language models (LLMs), or foundation models with respect to software engineering (SE) tasks.

Your PRIMARY focus is identifying whether the paper provides a structured model–task categorization.

========================
Inclusion Criteria
========================

IC1: The study investigates pre-trained models (PTMs), large language models (LLMs), or foundation models in a software engineering context.

IC2 (CORE CRITERION): The study provides an explicit or structured categorization or mapping between models and software engineering tasks. This includes:
- Explicit taxonomies or classifications
- Mapping models to SE tasks or SDLC phases
- Structured grouping of tasks linked to model capabilities

IC3 (Supporting): The study discusses model suitability, capabilities, or selection rationale for SE tasks across multiple tasks or categories.

========================
Exclusion Criteria
========================

EC1: The study focuses on non-software engineering domains (e.g., general NLP, computer vision, biomedical) without a clear SE context.

EC2: The study is a benchmark, dataset, or performance-only evaluation AND does not include any model–task categorization or structured mapping.

EC3: The study focuses only on educational or learning contexts (e.g., programming education, tutoring systems).

EC4: The study focuses on LLM-based agents or autonomous systems AND does not include model–task categorization.

========================
Decision Rules
========================

- The paper MUST satisfy IC1.
- The paper MUST satisfy IC2 to be included.
- If any exclusion criterion (EC1–EC4) is met, the paper must be excluded.

IMPORTANT:
- Papers that only list applications of models WITHOUT structuring or categorizing them should be EXCLUDED.
- Papers that evaluate models on tasks but do NOT organize or categorize the model–task relationship should be EXCLUDED.
- Prioritize explicit taxonomies, mappings, or structured groupings.

========================
Output Format
========================

Return ONLY a JSON object with:
- "included": boolean
- "reasoning": string

Paper title:
{title}

Abstract:
{abstract}
"""


    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an expert in classifying research papers based on provided criteria. "
                        "Respond only with a valid JSON object."
                    ),
                },
                {"role": "user", "content": criteria_prompt},
            ],
        )

        response_content = completion.choices[0].message.content
        result = json.loads(response_content)

        # Safety check in case model returns unexpected structure
        if not isinstance(result, dict):
            raise ValueError("Model response is not a JSON object.")

        result.setdefault("included", False)
        result.setdefault("reasoning", "No reasoning provided.")

        print(f"{title}: {result}")
        return result

    except Exception as e:
        print(f"Error processing paper '{title}': {e}")
        return {
            "included": False,
            "reasoning": f"API call failed: {e}"
        }

# Load CSV
try:
    df = pd.read_csv(csv_file_path)
    df = df.iloc[420:]
    len(df)
    print(df.iloc[0])
    print(f"Successfully loaded {len(df)} papers from {csv_file_path}.")
except FileNotFoundError:
    print(f"Error: The file '{csv_file_path}' was not found. Please check the path.")
    df = pd.DataFrame()

# Process papers
if not df.empty:
    missing_cols = [col for col in [title_column_name, abstract_column_name] if col not in df.columns]

    if missing_cols:
        print(f"Error: Missing column(s): {missing_cols}")
        print(f"Available columns: {df.columns.tolist()}")
    else:
        print("Starting paper evaluation using OpenAI API...")

        # Apply row by row
        df["evaluation_result"] = df.apply(
            lambda row: evaluate_paper(row[title_column_name], row[abstract_column_name]),
            axis=1
        )

        # Extract fields
        df["should_include"] = df["evaluation_result"].apply(lambda x: x.get("included", False))
        df["evaluation_reasoning"] = df["evaluation_result"].apply(lambda x: x.get("reasoning", ""))

        # Filter included papers
        filtered_papers = df[df["should_include"] == True]

        print(f"\nEvaluation complete. Found {len(filtered_papers)} papers matching the criteria.")
        print("\nFirst 5 filtered papers (with reasoning):")

        # In Colab/Jupyter this will display nicely
        display(filtered_papers[[title_column_name, abstract_column_name, "evaluation_reasoning"]].head())

        # Optional: save results
        output_path = "/content/drive/MyDrive/Preperation Phase/Papers_evaluated.csv"
        df.to_csv(output_path, index=False)
        print(f"\nSaved full results to: {output_path}")
else:
    print("No papers to process or DataFrame is empty.")

title        Comparing the adaptability of a Genetic Algori...
abstract     In the fast-paced world of software developmen...
published                                                 2024
link                          10.1109/ICDDS62937.2024.10910287
Source                                                  Scopus
Name: 420, dtype: object
Successfully loaded 165 papers from /content/drive/MyDrive/Preperation Phase/rr/cleaned_papers.csv.
Starting paper evaluation using OpenAI API...
Comparing the adaptability of a Genetic Algorithm and an LLM-based framework for Automated Software Test Data Generation: In the context of Web Applications: {'included': False, 'reasoning': 'The paper does not satisfy the core inclusion criterion (IC2) as it does not provide an explicit or structured model–task categorization or mapping between models and software engineering tasks. It compares the adaptability of two approaches for automated test data generation but does not offer a taxonomy, classification

,title,abstract,evaluation_reasoning
517,DAnTE: A Taxonomy for the Automation Degree of...,Software engineering researchers and practitio...,"The paper satisfies all inclusion criteria, pa..."
547,Proceedings - 2024 IEEE/ACM 3rd International ...,The proceedings contain 48 papers. The topics ...,The proceedings contain a paper with 'a taxono...



Saved full results to: /content/drive/MyDrive/Preperation Phase/Papers_evaluated.csv


In [ ]:
import os
import json
import pandas as pd
from openai import OpenAI
import bibtexparser

# --- User Configuration ---
bib_file_path = "/content/drive/MyDrive/Preperation Phase/acm.bib"
output_path = "/content/drive/MyDrive/Preperation Phase/Papers_evaluated_acm.csv"

# Use environment variable for safety:
# os.environ["OPENAI_API_KEY"] = "your-new-key-here"

client = OpenAI(api_key=openai_api_key)

def evaluate_paper(title, abstract):
    title = "" if title is None else str(title).strip()
    abstract = "" if abstract is None else str(abstract).strip()

    if not abstract:
        return {
            "included": False,
            "reasoning": "Abstract is empty or missing."
        }

    criteria_prompt = f"""You are an expert in software engineering research paper classification.
Your task is to evaluate a research paper's abstract and title against specific inclusion and exclusion criteria related to categorization of PTMs for software engineering tasks.

Here are the criteria:

Inclusion Criteria

IC1: The study investigates pre-trained models (PTMs), large language models (LLMs), or foundation models applied in a software engineering context.

IC2: The study provides an explicit or structured categorization of software engineering tasks and/or establishes a mapping between models and SE tasks (e.g., taxonomy, classification, or SDLC-based grouping).

IC3 (Optional): The study discusses model suitability, capabilities, or selection rationale for specific software engineering tasks.

Exclusion Criteria

EC1: The study focuses on non-software engineering domains (e.g., general NLP, computer vision, biomedical) without a clear SE context.

EC2: The study is a benchmark, dataset, or performance-only evaluation and does not provide any categorization or mapping between models and software engineering tasks.

EC3: The study focuses solely on software engineering tasks in learning or educational contexts (e.g., programming education, teaching tools).

EC4: The study focuses on LLM-based agents or autonomous systems without providing explicit model–task categorization or mapping.

Decision Rule:
- Include papers that satisfy IC1 AND (at least one of IC2 or IC3).
- Exclude papers that meet any of EC1, EC2, EC3, or EC4, even if they satisfy inclusion criteria.

Return ONLY a JSON object with:
- "included": boolean
- "reasoning": string

Paper title:
{title}

Abstract:
{abstract}
"""

    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an expert in classifying research papers based on provided criteria. "
                        "Respond only with a valid JSON object."
                    ),
                },
                {"role": "user", "content": criteria_prompt},
            ],
        )

        response_content = completion.choices[0].message.content
        result = json.loads(response_content)

        if not isinstance(result, dict):
            raise ValueError("Model response is not a JSON object.")

        result.setdefault("included", False)
        result.setdefault("reasoning", "No reasoning provided.")

        print(f"{title}: {result}")
        return result

    except Exception as e:
        print(f"Error processing paper '{title}': {e}")
        return {
            "included": False,
            "reasoning": f"API call failed: {e}"
        }

def load_bibtex_to_dataframe(bib_file_path):
    try:
        with open(bib_file_path, "r", encoding="utf-8") as bib_file:
            bib_database = bibtexparser.load(bib_file)

        papers = []
        for entry in bib_database.entries:
            papers.append({
                "entry_type": entry.get("ENTRYTYPE", ""),
                "citation_key": entry.get("ID", ""),
                "title": entry.get("title", ""),
                "abstract": entry.get("abstract", ""),
                "year": entry.get("year", ""),
                "author": entry.get("author", ""),
                "journal": entry.get("journal", ""),
                "booktitle": entry.get("booktitle", "")
            })

        df = pd.DataFrame(papers)
        print(f"Successfully loaded {len(df)} papers from {bib_file_path}.")
        return df

    except FileNotFoundError:
        print(f"Error: The file '{bib_file_path}' was not found.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error reading BibTeX file: {e}")
        return pd.DataFrame()

# Load BibTeX
df = load_bibtex_to_dataframe(bib_file_path)

# Process papers
if not df.empty:
    if "title" not in df.columns or "abstract" not in df.columns:
        print("Error: title or abstract field missing.")
        print(f"Available columns: {df.columns.tolist()}")
    else:
        print("Starting paper evaluation using OpenAI API...")

        df["evaluation_result"] = df.apply(
            lambda row: evaluate_paper(row["title"], row["abstract"]),
            axis=1
        )

        df["should_include"] = df["evaluation_result"].apply(lambda x: x.get("included", False))
        df["evaluation_reasoning"] = df["evaluation_result"].apply(lambda x: x.get("reasoning", ""))

        filtered_papers = df[df["should_include"] == True]

        print(f"\nEvaluation complete. Found {len(filtered_papers)} papers matching the criteria.")
        print("\nFirst 5 filtered papers (with reasoning):")

        try:
            from IPython.display import display
            display(filtered_papers[["title", "abstract", "evaluation_reasoning"]].head())
        except ImportError:
            print(filtered_papers[["title", "abstract", "evaluation_reasoning"]].head())

        df.to_csv(output_path, index=False)
        print(f"\nSaved full results to: {output_path}")
else:
    print("No papers to process or DataFrame is empty.")

Successfully loaded 76 papers from /content/drive/MyDrive/Preperation Phase/acm.bib.
Starting paper evaluation using OpenAI API...
Large Language Model-Based Agents for Software Engineering: A Survey: {'included': False, 'reasoning': 'The paper explicitly focuses on LLM-based agents applied to software engineering, which meets inclusion criterion IC1. However, it also falls under exclusion criterion EC4, as it focuses on LLM-based agents in SE. Therefore, despite satisfying part of the inclusion criteria, the paper is excluded based on the decision rule.'}
Embedding Traceability in Large Language Model Code Generation: Towards Trustworthy AI-Augmented Software Engineering: {'included': False, 'reasoning': 'The paper focuses on embedding traceability in LLM-based code generation, which falls under the exclusion criterion EC4 as it involves LLM-based agents in SE (software engineering) tasks. While it deals with LLMs in the context of software engineering, the emphasis is not on categori

,title,abstract,evaluation_reasoning
2,Automated categorization of pre-trained models...,Software engineering (SE) activities have been...,The paper discusses PTMs applied to software e...



Saved full results to: /content/drive/MyDrive/Preperation Phase/Papers_evaluated_acm.csv


In [ ]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 11.7 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
from pathlib import Path
from typing import Any, Dict, List, Optional

from openai import OpenAI
from pypdf import PdfReader


# ----------------------------
# Configuration
# ----------------------------

MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1")
MAX_CHARS = 50000   # trim long papers to control cost
TEMPERATURE = 0     # deterministic screening


# ----------------------------
# JSON Schema
# ----------------------------

SCREEN_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "paper_title": {"type": "string"},
        "include_for_review": {
            "type": "string",
            "enum": ["yes", "no", "maybe"]
        },
        "relevance_score": {
            "type": "integer",
            "minimum": 0,
            "maximum": 100
        },
        "screening_reason": {"type": "string"},
        "task_scope": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "label": {"type": "string"},
                "details": {"type": "string"}
            },
            "required": ["label", "details"]
        },
        "model_type": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "label": {"type": "string"},
                "details": {"type": "string"}
            },
            "required": ["label", "details"]
        },
        "relationship_type": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "label": {
                    "type": "string",
                    "enum": [
                        "capability-based",
                        "designed-for",
                        "mapping-based",
                        "categorization-based",
                        "selection-based",
                        "other"
                    ]
                },
                "details": {"type": "string"}
            },
            "required": ["label", "details"]
        },
        "granularity_level": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "label": {
                    "type": "string",
                    "enum": ["SDLC-level", "task-level", "sub-task-level", "mixed", "unclear"]
                },
                "details": {"type": "string"}
            },
            "required": ["label", "details"]
        },
        "categorization_approach": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "label": {
                    "type": "string",
                    "enum": ["explicit taxonomy", "implicit grouping", "no categorization", "unclear"]
                },
                "details": {"type": "string"}
            },
            "required": ["label", "details"]
        },
        "se_coverage": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "label": {
                    "type": "string",
                    "enum": ["full SDLC", "partial SDLC", "single SE task", "SE activity", "non-SE/adjacent", "unclear"]
                },
                "details": {"type": "string"}
            },
            "required": ["label", "details"]
        },
        "methodology": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "label": {
                    "type": "string",
                    "enum": [
                        "literature review",
                        "systematic literature review",
                        "systematic mapping study",
                        "empirical study",
                        "benchmark study",
                        "automated analysis",
                        "manual annotation",
                        "case study",
                        "mixed",
                        "other",
                        "unclear"
                    ]
                },
                "details": {"type": "string"}
            },
            "required": ["label", "details"]
        },
        "evidence_type": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "label": {
                    "type": "string",
                    "enum": [
                        "conceptual",
                        "experimental",
                        "dataset-based",
                        "mixed",
                        "other",
                        "unclear"
                    ]
                },
                "details": {"type": "string"}
            },
            "required": ["label", "details"]
        },
        "model_selection_support": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "label": {
                    "type": "string",
                    "enum": ["yes", "limited", "no", "unclear"]
                },
                "details": {"type": "string"}
            },
            "required": ["label", "details"]
        },
        "automation_level": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "label": {
                    "type": "string",
                    "enum": ["manual", "semi-automated", "automated", "unclear"]
                },
                "details": {"type": "string"}
            },
            "required": ["label", "details"]
        },
        "exclusion_flags": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": [
                    "non-SE focus",
                    "benchmark-only",
                    "performance-only",
                    "agent-focused",
                    "code-learning-only",
                    "not-english",
                    "no PTM-task relation discussion",
                    "unclear"
                ]
            }
        },
        "evidence_snippets": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 2,
            "maxItems": 6
        },
        "confidence": {
            "type": "integer",
            "minimum": 0,
            "maximum": 100
        }
    },
    "required": [
        "paper_title",
        "include_for_review",
        "relevance_score",
        "screening_reason",
        "task_scope",
        "model_type",
        "relationship_type",
        "granularity_level",
        "categorization_approach",
        "se_coverage",
        "methodology",
        "evidence_type",
        "model_selection_support",
        "automation_level",
        "exclusion_flags",
        "evidence_snippets",
        "confidence"
    ]
}


# ----------------------------
# Prompt
# ----------------------------

SYSTEM_PROMPT = """
You are an expert research screener for a rapid literature review on how prior studies conceptualize
relationships between pre-trained models (PTMs), large language models (LLMs), foundation models,
and software engineering (SE) tasks.

Your job:
1. Read the paper text carefully.
2. Extract the requested fields conservatively.
3. Use only evidence grounded in the provided paper text.
4. If something is not clearly supported, mark it as "unclear".
5. Classify based on the paper’s main contribution, not isolated statements.

----------------------------------------
CORE CONCEPTUAL FRAMEWORK
----------------------------------------

You must extract TWO separate dimensions:

----------------------------------------
1. relationship_type (SEMANTIC MEANING)
----------------------------------------

This describes WHAT kind of relationship exists between models and SE tasks.

Choose ONE:

- capability-based:
  The paper describes what PTMs/LLMs are capable of doing for SE tasks.
  Focus is on abilities, applications, or potential use, often based on:
  - model architecture
  - general model behavior
  - empirical capability observations

  Examples:
  - “LLMs can support bug fixing, testing, and summarization.”
  - “Transformer-based models can perform code understanding tasks.”

  Important:
  - Use this when the paper discusses what models CAN DO.
  - If tasks are not explicitly assigned as intended use, prefer capability-based.

- designed-for:
  The paper associates models with specific SE tasks as intended use.

  This includes BOTH:

  (1) Explicit model design or adaptation:
      - models built, pre-trained, or fine-tuned for tasks

  (2) Intended-task association derived from metadata or external sources:
      - model cards
      - task tags (e.g., Hugging Face)
      - repository labels
      - keyword-based matching of model descriptions to tasks

  Examples:
  - “The model is fine-tuned for vulnerability detection.”
  - “We assign tasks to models using Hugging Face task tags.”
  - “Models are labeled for code summarization and testing.”

  Important:
  - If the paper links models to tasks via tags, metadata, or model cards → choose designed-for.
  - This represents intended use, even if not architecturally enforced.

- selection-based:
  The paper helps decide which model is most suitable for a given SE task.
  The emphasis is on recommendation, comparison, ranking, or suitability.

  Examples:
  - “We recommend Model A for bug detection.”
  - “This study identifies the best PTM for testing tasks.”

  Important:
  - Requires decision-support or recommendation logic.
  - If only describing abilities → NOT selection-based.

----------------------------------------
2. structuring_approach (METHOD)
----------------------------------------

This describes HOW the paper organizes or represents model–task relationships.

Choose ONE:

- mapping-based:
  The paper explicitly constructs or extracts links between models and SE tasks.

  This includes:
  - model–task mappings
  - matrices or datasets
  - Hugging Face / repository / model card linking
  - keyword-based or tag-based matching
  - automated or manual extraction

  Examples:
  - “We map PTMs to SE tasks using model cards.”
  - “We construct a model–task dataset.”

- categorization-based:
  The paper groups tasks, models, or studies into categories without explicit model–task links.

  Examples:
  - “Tasks are grouped into SDLC phases.”
  - “We categorize studies by task type.”

- none:
  No explicit structuring of model–task relationships.

----------------------------------------
IMPORTANT DISTINCTIONS
----------------------------------------

- relationship_type = meaning (what kind of relation exists)
- structuring_approach = method (how it is organized)

- mapping-based is NOT a relationship_type
- categorization-based is NOT a relationship_type

----------------------------------------
HOW TO RESOLVE AMBIGUITY
----------------------------------------

- If tasks are linked via model cards, tags, or metadata → choose designed-for.
- If tasks are discussed as general abilities → choose capability-based.
- If the paper recommends or ranks models → choose selection-based.
- If the paper builds explicit model-task links → mapping-based.
- If the paper only groups tasks/models → categorization-based.
- If unsure → use "unclear" and be conservative.

----------------------------------------
FIELD DEFINITIONS
----------------------------------------

Extract the following fields:

- task_scope: SE tasks considered (e.g., bug fixing, testing, SDLC-level)
- model_type: PTM, LLM, foundation model, code-specific model, etc.
- granularity_level: SDLC-level, task-level, sub-task-level, mixed, unclear
- categorization_approach: explicit taxonomy, implicit grouping, no categorization, unclear
- relationship_type: capability-based, designed-for, selection-based, unclear
- structuring_approach: mapping-based, categorization-based, none, unclear
- se_coverage: full SDLC, partial SDLC, single SE task, SE activity, non-SE/adjacent, unclear
- methodology: literature review, empirical study, automated analysis, etc.
- evidence_type: conceptual, experimental, dataset-based, mixed, unclear
- model_selection_support: yes, limited, no, unclear
- automation_level: manual, semi-automated, automated, unclear

----------------------------------------
GENERAL RULE
----------------------------------------

Classify using the paper’s primary analytical contribution.
Do not rely on background discussion.

----------------------------------------
OUTPUT
----------------------------------------

Return valid JSON only.
Do not include any explanations outside the JSON.
"""

USER_TEMPLATE = """
Analyze the following paper for PTM–SE screening.

Filename: {filename}

----------------------------------------
INSTRUCTIONS
----------------------------------------

- Carefully read the provided paper text.
- Extract all required fields strictly based on the SYSTEM definitions.
- Do NOT guess. If information is not clearly supported, use "unclear".
- Focus on the paper’s MAIN contribution, not background or related work.

----------------------------------------
KEY TASK
----------------------------------------

You MUST determine:

1. relationship_type (semantic meaning):
   - capability-based
   - designed-for
   - selection-based
   - unclear

2. structuring_approach (method):
   - mapping-based
   - categorization-based
   - none
   - unclear

----------------------------------------
OUTPUT REQUIREMENTS
----------------------------------------

- Return ONLY valid JSON.
- Follow the predefined schema strictly.
- Use the exact labels provided in the definitions.
- Include short justifications in "details" fields where applicable.
- Be conservative and avoid over-interpretation.

----------------------------------------
PAPER TEXT
----------------------------------------

\"\"\"
{text}
\"\"\"
"""


# ----------------------------
# Helpers
# ----------------------------

def extract_text_from_pdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        try:
            txt = page.extract_text() or ""
        except Exception:
            txt = ""
        if txt.strip():
            pages.append(f"\n\n--- Page {i+1} ---\n{txt}")
    return "\n".join(pages)


def normalize_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def select_relevant_passages(text: str, max_chars: int = MAX_CHARS) -> str:
    """
    Keep title/abstract-ish beginning plus sections likely to contain the needed signals.
    Falls back to truncation if no headings are found.
    """
    lower = text.lower()

    heading_patterns = [
        "abstract", "introduction", "method", "methodology",
        "approach", "results", "discussion", "conclusion",
        "survey", "review", "taxonomy", "mapping", "software engineering",
        "task", "benchmark", "dataset", "model", "foundation model", "llm"
    ]

    # Always keep the beginning
    chunks = [text[:12000]]

    # Pull windows around important keywords
    for pat in heading_patterns:
        for m in re.finditer(re.escape(pat), lower):
            start = max(0, m.start() - 1200)
            end = min(len(text), m.start() + 3500)
            chunks.append(text[start:end])

    merged = "\n\n".join(chunks)
    merged = normalize_text(merged)

    # Hard trim
    return merged[:max_chars]


def make_client() -> OpenAI:

    api_key = openai_api_key
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY is not set.")
    return OpenAI(api_key=api_key)


def judge_paper_text(text: str, filename: str, model: str = MODEL) -> Dict[str, Any]:
    client = make_client()

    prompt = USER_TEMPLATE.format(
        filename=filename,
        text=text
    )

    response = client.responses.create(
        model=model,
        temperature=TEMPERATURE,
        input=[
            {
                "role": "system",
                "content": [{"type": "input_text", "text": SYSTEM_PROMPT}],
            },
            {
                "role": "user",
                "content": [{"type": "input_text", "text": prompt}],
            },
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "ptm_se_screening",
                "schema": SCREEN_SCHEMA,
                "strict": True,
            }
        },
    )

    # The SDK aggregates text outputs in output_text
    raw = response.output_text
    return json.loads(raw)


def screen_pdf(pdf_path: str, model: str = MODEL) -> Dict[str, Any]:
    pdf_path = str(pdf_path)
    raw_text = extract_text_from_pdf(pdf_path)

    if not raw_text.strip():
        raise ValueError(f"No extractable text found in: {pdf_path}")

    selected_text = select_relevant_passages(raw_text, max_chars=MAX_CHARS)
    result = judge_paper_text(
        text=selected_text,
        filename=Path(pdf_path).name,
        model=model
    )
    result["source_pdf"] = pdf_path
    return result


def save_result(result: Dict[str, Any], output_dir: str = "screening_results") -> str:
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    stem = Path(result["source_pdf"]).stem
    out_path = Path(output_dir) / f"{stem}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    return str(out_path)


def print_human_summary(result: Dict[str, Any]) -> None:
    print("=" * 80)
    print("TITLE:", result.get("paper_title"))
    print("INCLUDE:", result.get("include_for_review"))
    print("RELEVANCE:", result.get("relevance_score"))
    print("RELATIONSHIP:", result.get("relationship_type", {}).get("label"))
    print("TASK SCOPE:", result.get("task_scope", {}).get("label"))
    print("MODEL TYPE:", result.get("model_type", {}).get("label"))
    print("GRANULARITY:", result.get("granularity_level", {}).get("label"))
    print("CATEGORIZATION:", result.get("categorization_approach", {}).get("label"))
    print("SE COVERAGE:", result.get("se_coverage", {}).get("label"))
    print("METHODOLOGY:", result.get("methodology", {}).get("label"))
    print("EVIDENCE TYPE:", result.get("evidence_type", {}).get("label"))
    print("MODEL SELECTION SUPPORT:", result.get("model_selection_support", {}).get("label"))
    print("AUTOMATION LEVEL:", result.get("automation_level", {}).get("label"))
    print("CONFIDENCE:", result.get("confidence"))
    print("REASON:", result.get("screening_reason"))
    print("EVIDENCE:")
    for snip in result.get("evidence_snippets", []):
        print(f" - {snip}")
    print("=" * 80)


if __name__ == "__main__":
    pdf_path = "/content/AI-Driven_Software_Test_Automation_An_AI4SE-Oriented_Survey_of_Techniques_Tools_and_Challenges.pdf"
    result = screen_pdf(pdf_path, model=MODEL)
    print_human_summary(result)
    print(result)

TITLE: AI-Driven Software Test Automation: An AI4SE-Oriented Survey of Techniques, Tools, and Challenges
INCLUDE: yes
RELEVANCE: 9
RELATIONSHIP: capability-based
TASK SCOPE: task-level
MODEL TYPE: PTM, LLM, code-specific model, other AI models
GRANULARITY: task-level
CATEGORIZATION: explicit taxonomy
SE COVERAGE: partial SDLC
METHODOLOGY: systematic literature review
EVIDENCE TYPE: mixed
MODEL SELECTION SUPPORT: no
AUTOMATION LEVEL: manual
CONFIDENCE: 9
REASON: The paper systematically surveys the use of AI models, including LLMs and PTMs, for software testing tasks, and constructs a lifecycle-oriented taxonomy mapping AI techniques to testing activities.
EVIDENCE:
 - We offer a systematic overview of 76 industrial tools and peer-reviewed research and examine the contribution of Machine Learning, Deep Learning, Large Language Models, Reinforcement Learning, and other paradigms of artificial intelligence to modern test automation pipelines.
 - The paper suggests a lifecycle-oriented tax

In [ ]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 5.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from rapidfuzz import fuzz
import ast

# Load CSV
df = pd.read_csv("/content/drive/MyDrive/Preperation Phase/rr/Papers_evaluated - Papers_evaluated.csv")

# Ensure titles are strings
df["title"] = df["title"].astype(str)

threshold = 90

kept_indices = []
kept_titles = []

for idx, title in df["title"].items():
    is_duplicate = False

    for kept_title in kept_titles:
        similarity = fuzz.ratio(title.lower(), kept_title.lower())
        if similarity >= threshold:
            is_duplicate = True
            break

    if not is_duplicate:
        kept_indices.append(idx)
        kept_titles.append(title)

# Keep first occurrence of fuzzy-duplicate titles
clean_df = df.loc[kept_indices].reset_index(drop=True)

# Fill missing evaluation_reasoning from evaluation_result
for i, row in clean_df.iterrows():
    reasoning_val = row.get("evaluation_reasoning", None)

    if pd.isna(reasoning_val) or str(reasoning_val).strip() == "":
        result = row.get("evaluation_result", None)

        if pd.notna(result) and str(result).strip() != "":
            try:
                parsed = ast.literal_eval(result)   # parses Python dict string
                if isinstance(parsed, dict):
                    clean_df.at[i, "evaluation_reasoning"] = parsed.get("reasoning", "")
            except Exception as e:
                print(f"Could not parse evaluation_result at row {i}: {e}")

# Save
clean_df.to_csv("cleaned_papers.csv", index=False)

print(f"Original rows: {len(df)}")
print(f"Cleaned rows: {len(clean_df)}")

Original rows: 756
Cleaned rows: 693


In [ ]:
pip install rapidfuzz


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 71.5 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
from pathlib import Path
from typing import List, Dict

from openai import OpenAI
from pypdf import PdfReader

# =========================
# CONFIG
# =========================

MODEL = "gpt-4.1"

INPUT_DIR = "/content/drive/MyDrive/Preperation Phase/papers/analyze"
OUTPUT_DIR = "/content/drive/MyDrive/Preperation Phase/papers/analyze"

PER_PAPER_OUTPUT = os.path.join(OUTPUT_DIR, "per_paper_analysis.json")
FINAL_SYNTHESIS_OUTPUT = os.path.join(OUTPUT_DIR, "final_synthesis.md")

MAX_CHARS_PER_PAPER = 120000

# =========================
# PROMPTS
# =========================

SYSTEM_PROMPT = """
You are an expert researcher in software engineering, machine learning, and empirical research design.
You help design rigorous thesis-level research questions from literature on pre-trained models, categorization studies, and ecosystem analysis.
Be analytical, structured, specific, and concise.
"""

PER_PAPER_TEMPLATE = """
I will provide you with one academic paper related to the categorization, taxonomy, survey, benchmark analysis, or ecosystem analysis of pre-trained models (PTMs), foundation models, or language models, especially for software engineering tasks.

My goal is to design empirical research questions for a knowledge graph-based analysis of the Hugging Face ecosystem.

Context about my knowledge graph:
- It contains software-engineering-related models (SE models) and also non-SE models.
- It includes model-to-model lineage and reuse relations such as: fine-tuned-from, adapted-from, quantized-from, merged-from, adapter-based reuse, and ancestor/descendant relations.
- It includes tasks, benchmarks, datasets, papers, collections, spaces, users, and organizations.
- Some benchmarks from prior studies (e.g., Gonzalez et al.) are mapped to specific models.
- I want to understand the ecosystem of SE-related models and later support a model-selection/recommendation system.

Some example directions I may investigate:
- Structural:
  - Do SE models form distinct clusters in the model reuse graph?
  - Are SE models deeper in fine-tuning chains than general models?
- Evolutionary:
  - What base models are most frequently adapted for SE tasks?
  - Do SE models rely more on adapters vs full fine-tuning?
- Social:
  - Are SE models developed by smaller specialized communities?
  - Is model popularity correlated with reuse depth?
- Benchmark-driven:
  - Do models evaluated on SE benchmarks show different reuse patterns?

Your task is to analyze the paper and help me derive empirical research opportunities from it.

Please do the following carefully:

1. Identify the main categorization dimensions used in the paper.
2. Extract the explicit or implicit assumptions behind the categorization.
3. Identify limitations, blind spots, or gaps mentioned in the paper.
4. Extract or infer future research directions.
5. Based on the paper, generate 3 to 5 concrete empirical research questions that could be answered using my knowledge graph.
6. For each research question, also provide:
   - required entities
   - required relationships
   - suggested analysis type
   - measurable variables / metrics
   - expected insight or contribution
7. If possible, map the paper's perspective into one or more of these analytical lenses:
   - structural
   - evolutionary
   - social
   - benchmark-driven
   - metadata/artefact-driven

Important instructions:
- Do not just restate the paper.
- Prioritize questions that go beyond the paper’s original categorization and use the paper as inspiration for ecosystem analysis.
- Prefer graph-oriented and comparative questions over generic survey questions.
- Make the questions suitable for a thesis in empirical software engineering.
- Be concrete and avoid vague phrases.

Return the result in valid JSON with this exact structure:

{
  "paper_summary": "...",
  "paper_type": "...",
  "categorization_dimensions": [
    {
      "dimension": "...",
      "description": "...",
      "evidence_from_paper": "..."
    }
  ],
  "assumptions": ["...", "..."],
  "limitations_or_gaps": ["...", "..."],
  "future_directions": ["...", "..."],
  "analytical_lenses": [
    {
      "lens": "structural | evolutionary | social | benchmark-driven | metadata/artefact-driven",
      "relevance": "..."
    }
  ],
  "empirical_questions": [
    {
      "question": "...",
      "motivation_from_paper": "...",
      "entities": ["...", "..."],
      "relationships": ["...", "..."],
      "analysis_type": "...",
      "metrics": ["...", "..."],
      "expected_insight": "..."
    }
  ]
}

Paper title: {title}

Paper text:
\"\"\"
{paper_text}
\"\"\"
"""

FINAL_SYNTHESIS_TEMPLATE = """
I will provide analyses extracted from multiple academic papers that categorize, survey, benchmark, or otherwise analyze pre-trained models for software engineering tasks.

Using these analyses, synthesize the literature and help define strong thesis-level research directions for a knowledge graph-based study of the Hugging Face ecosystem.

My study context:
- I am building a knowledge graph of software-engineering-related models in Hugging Face.
- The graph includes SE models and non-SE models, plus lineage/reuse relations, tasks, datasets, benchmarks, papers, users, organizations, collections, and spaces.
- My immediate goal is to define strong empirical research questions for understanding the ecosystem.
- My longer-term goal is to support a model selection / recommendation approach based on this graph.

Please produce:

1. Key recurring categorization dimensions across papers
2. Common assumptions across papers
3. Repeated limitations/gaps in the literature
4. Repeated future research opportunities
5. A consolidated set of 5 to 10 high-quality thesis-suitable research questions

For each final research question:
- explain why it matters
- explain how it connects to literature gaps
- explain what graph data is needed
- explain what analysis approach is appropriate
- explain what kind of finding would make the question valuable

6. Organize the final research questions into thematic groups.
7. Propose a compact empirical analysis catalog for my thesis.
8. Suggest which ontology/schema dimensions are essential in the first version of the knowledge graph and which can be added later.

Return the result in clear structured markdown with these sections:

# Recurring Categorization Dimensions
# Common Assumptions
# Repeated Limitations and Gaps
# Repeated Future Research Opportunities
# Thesis-Suitable Research Questions
# Empirical Analysis Catalog
# Recommended Knowledge Graph Schema Priorities

Here are the extracted analyses:

{all_analyses}
"""

# =========================
# CLIENT
# =========================

def get_client() -> OpenAI:
    if not API_KEY:
        raise ValueError(
            "OPENAI_API_KEY is not set. Example:\n"
            "export OPENAI_API_KEY='your_key_here'"
        )
    return OpenAI(api_key=API_KEY)

# =========================
# PDF EXTRACTION
# =========================

def extract_text_from_pdf(pdf_path: Path) -> str:
    """
    Extract text from a text-based PDF using pypdf.
    """
    reader = PdfReader(str(pdf_path))
    pages_text = []

    for page_num, page in enumerate(reader.pages, start=1):
        try:
            text = page.extract_text() or ""
        except Exception as e:
            print(f"Warning: failed to extract page {page_num} from {pdf_path.name}: {e}")
            text = ""

        if text.strip():
            pages_text.append(f"\n--- PAGE {page_num} ---\n{text}")

    return "\n".join(pages_text).strip()

def load_pdf_files(input_dir: str) -> List[Dict[str, str]]:
    papers = []

    for path in sorted(Path(input_dir).glob("*.pdf")):
        print(f"Reading PDF: {path.name}")
        text = extract_text_from_pdf(path)

        if not text.strip():
            print(f"Warning: No extractable text found in {path.name}")
            continue

        if len(text) > MAX_CHARS_PER_PAPER:
            text = text[:MAX_CHARS_PER_PAPER]

        papers.append({
            "title": path.stem,
            "text": text,
            "filename": str(path)
        })

    return papers

# =========================
# OPENAI CALLS
# =========================

def call_model_json(client: OpenAI, system_prompt: str, user_prompt: str) -> dict:
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    content = response.choices[0].message.content
    return json.loads(content)

def call_model_text(client: OpenAI, system_prompt: str, user_prompt: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

# =========================
# PIPELINE
# =========================

def analyze_single_paper(client: OpenAI, title: str, paper_text: str) -> dict:
    prompt = PER_PAPER_TEMPLATE.format(title=title, paper_text=paper_text)
    return call_model_json(client, SYSTEM_PROMPT, prompt)

def analyze_all_papers(client: OpenAI, papers: List[Dict[str, str]]) -> List[Dict]:
    results = []

    for idx, paper in enumerate(papers, start=1):
        print(f"[{idx}/{len(papers)}] Analyzing: {paper['title']}")
        try:
            result = analyze_single_paper(client, paper["title"], paper["text"])
            result["_meta"] = {
                "title": paper["title"],
                "filename": paper["filename"]
            }
            results.append(result)
            time.sleep(1)
        except Exception as exc:
            print(f"Error while analyzing {paper['title']}: {exc}")
            results.append({
                "_meta": {
                    "title": paper["title"],
                    "filename": paper["filename"],
                    "error": str(exc)
                }
            })

    return results

def synthesize_all(client: OpenAI, analyses: List[Dict]) -> str:
    serializable = json.dumps(analyses, indent=2, ensure_ascii=False)
    prompt = FINAL_SYNTHESIS_TEMPLATE.format(all_analyses=serializable)
    return call_model_text(client, SYSTEM_PROMPT, prompt)

# =========================
# SAVE HELPERS
# =========================

def ensure_output_dir() -> None:
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

def save_json(data, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def save_text(text: str, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

# =========================
# MAIN
# =========================

def main():
    ensure_output_dir()
    client = get_client()

    papers = load_pdf_files(INPUT_DIR)
    if not papers:
        raise FileNotFoundError(
            f"No readable PDF files found in '{INPUT_DIR}'."
        )

    print(f"Found {len(papers)} readable PDFs.")

    per_paper_results = analyze_all_papers(client, papers)
    save_json(per_paper_results, PER_PAPER_OUTPUT)
    print(f"Saved per-paper analysis to: {PER_PAPER_OUTPUT}")

    synthesis = synthesize_all(client, per_paper_results)
    save_text(synthesis, FINAL_SYNTHESIS_OUTPUT)
    print(f"Saved final synthesis to: {FINAL_SYNTHESIS_OUTPUT}")

    print("Done.")

if __name__ == "__main__":
    main()

Reading PDF: A Survey on Large Language Models for Software Engineering.pdf
Reading PDF: Automated categorization of pre-trained models for software.pdf
Reading PDF: How do Pre-Trained Models Support Software.pdf
Reading PDF: Large Language Models for Software Engineering A Systematic Literature Review.pdf
Found 4 readable PDFs.
[1/4] Analyzing: A Survey on Large Language Models for Software Engineering
Error while analyzing A Survey on Large Language Models for Software Engineering: '\n  "paper_summary"'
[2/4] Analyzing: Automated categorization of pre-trained models for software
Error while analyzing Automated categorization of pre-trained models for software: '\n  "paper_summary"'
[3/4] Analyzing: How do Pre-Trained Models Support Software
Error while analyzing How do Pre-Trained Models Support Software: '\n  "paper_summary"'
[4/4] Analyzing: Large Language Models for Software Engineering A Systematic Literature Review
Error while analyzing Large Language Models for Software Enginee

In [ ]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 5.4 MB/s eta 0:00:00
